# Deep Dive into Logistic Regression: Foundations, Proofs, and Evaluation

This notebook provides a rigorous, consolidated breakdown of logistic regression, covering mathematical origins, parameter interpretations, optimization proofs, loss landscape analysis, thresholding dynamics, and classification evaluation frameworks.

## 1. The Origin & Anatomy of the Sigmoid Function

### 1.1 Mathematical Derivation: Combining $1/x$ and $e^x$
To map real-valued linear outputs $z \in (-\infty, +\infty)$ into bounded probabilities $\hat{y} \in (0, 1)$, we synthesize an exponential transformation with a hyperbolic/reciprocal structure:

1. **Odds Ratio**: The ratio of probability of occurrence ($p$) to non-occurrence ($1-p$):
   $$\text{Odds} = \frac{p}{1-p} \in (0, +\infty)$$
2. **Exponential Mapping**: Map unbounded $z \in (-\infty, +\infty)$ into non-negative odds space via $e^z$:
   $$\frac{p}{1-p} = e^z$$
3. **Algebraic Inversion**: Isolate probability $p$:
   $$p = e^z(1 - p) = e^z - p e^z \implies p(1 + e^z) = e^z \implies p = \frac{e^z}{1 + e^z}$$
4. **Standard Sigmoid Form**: Divide numerator and denominator by $e^z$:
   $$p = \frac{1}{1 + e^{-z}}$$

#### Theoretical Foundations:
* **Why $e^z$?**: It is a smooth, monotonic bijection mapping $(-\infty, +\infty) \to (0, +\infty)$. It yields a log-linear relationship where a unit change in feature $x_i$ acts multiplicatively on odds by $e^{w_i}$. Furthermore, it functions as the canonical link function for Bernoulli distributions within Generalized Linear Models (GLMs).
* **Role of the Reciprocal/Hyperbola ($1/x$)**: The form $\frac{1}{1 + X}$ represents a translated reciprocal curve $f(X) = \frac{1}{X}$ evaluated at $X = 1 + e^{-z}$. It compresses the infinitely expanding exponential range $(0, +\infty)$ into a strictly bounded interval $(0, 1)$.

### 1.2 Key Definitions & Alternative S-Shaped Functions
* **Sigmoid**: Literally "S-shaped". Any continuous, monotonic function with horizontal asymptotes as $z \to \pm\infty$.
* **Asymptotic Boundaries $(0, 1)$**:
  $$\lim_{z \to +\infty} \sigma(z) = \lim_{z \to +\infty} \frac{1}{1 + e^{-z}} = \frac{1}{1 + 0} = 1$$
  $$\lim_{z \to -\infty} \sigma(z) = \lim_{z \to -\infty} \frac{1}{1 + e^{-z}} = \frac{1}{1 + \infty} = 0$$
* **Alternative Activation Functions**:
  1. **Probit**: $\Phi(z) = \int_{-\infty}^z \frac{1}{\sqrt{2\pi}} e^{-t^2/2} dt$ (CDF of the standard Gaussian distribution).
  2. **Hyperbolic Tangent**: $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}} \in (-1, 1)$, related via $\sigma(z) = \frac{\tanh(z/2) + 1}{2}$.
  3. **Arctan**: $f(z) = \frac{1}{\pi} \arctan(z) + \frac{1}{2}$.
  4. **Algebraic Sigmoid**: $f(z) = \frac{z}{\sqrt{1 + z^2}}$.

## 2. Linear Component & Log Odds

### 2.1 Formulation
$$z = b + w_1 x_1 + w_2 x_2 + \dots + w_n x_n = \mathbf{w}^T \mathbf{x} + b$$

### 2.2 Why $z$ is the Log Odds
Let $p = P(y=1|\mathbf{x}) = \hat{y} = \sigma(z)$. Evaluating the odds ratio:
$$\frac{p}{1 - p} = \frac{\frac{1}{1 + e^{-z}}}{1 - \frac{1}{1 + e^{-z}}} = \frac{\frac{1}{1 + e^{-z}}}{\frac{e^{-z}}{1 + e^{-z}}} = \frac{1}{e^{-z}} = e^z$$
Taking the natural logarithm:
$$\ln\left(\frac{p}{1 - p}\right) = \ln(e^z) = z$$
Hence, $z$ directly computes the **log-odds** (logit) of the positive class.

### 2.3 Evaluation vs. Optimization Dynamics
* **Inference (Forward Pass)**: $z$ is evaluated as a discrete scalar **value** from inputs $\mathbf{x}$ and mapped via $\sigma(z) \in (0, 1)$.
* **Optimization (Backward Pass)**: $z$ is treated as an algebraic **equation** (linear combination of parameters) allowing analytic gradients $\frac{\partial L}{\partial \mathbf{w}} = \frac{\partial L}{\partial \hat{y}} \frac{\partial \hat{y}}{\partial z} \frac{\partial z}{\partial \mathbf{w}}$.

## 3. Mathematical Proofs

### Proof 1: Incremental Change in Linear Model
**Proposition**: For the model $y = b + 3x_1$, incrementing $x_1$ by $1$ increases $y$ by $3$.

* **Proof**:
  Let $y_{t} = b + 3x_1$.
  Let $x_{1}' = x_1 + 1$.
  $$y_{t+1} = b + 3(x_1 + 1) = b + 3x_1 + 3$$
  $$\Delta y = y_{t+1} - y_t = (b + 3x_1 + 3) - (b + 3x_1) = 3 \quad \blacksquare$$

---

### Proof 2: Mean Log Loss Decouples Batch Size and Learning Rate
**Proposition**: Formulating empirical risk as Mean Log Loss decouples learning rate $\eta$ from mini-batch size $N$.

* **Sum/Total Loss Formulation**:
  $$L_{total}(\mathbf{w}) = \sum_{i=1}^N \ell_i(\mathbf{w}) \implies \nabla_\mathbf{w} L_{total}(\mathbf{w}) = \sum_{i=1}^N \nabla_\mathbf{w} \ell_i(\mathbf{w})$$
  $$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \eta \sum_{i=1}^N \nabla_\mathbf{w} \ell_i(\mathbf{w})$$
  The gradient step scales proportionally with batch size $N$. Changing batch size requires re-scaling $\eta$.

* **Mean Loss Formulation**:
  $$L_{mean}(\mathbf{w}) = \frac{1}{N} \sum_{i=1}^N \ell_i(\mathbf{w}) \implies \nabla_\mathbf{w} L_{mean}(\mathbf{w}) = \frac{1}{N} \sum_{i=1}^N \nabla_\mathbf{w} \ell_i(\mathbf{w})$$
  $$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \eta \left( \frac{1}{N} \sum_{i=1}^N \nabla_\mathbf{w} \ell_i(\mathbf{w}) \right)$$
  The quantity $\frac{1}{N} \sum_{i=1}^N \nabla_\mathbf{w} \ell_i(\mathbf{w})$ is a Monte Carlo estimator of the true population gradient $\mathbb{E}[\nabla_\mathbf{w} \ell(\mathbf{w})]$. Its magnitude remains scale-invariant to $N$, allowing learning rate $\eta$ to be tuned independently of batch size. $\blacksquare$

---

### Proof 3: Asymptotic Collapse Without Regularization
**Proposition**: On linearly separable data, unconstrained log loss minimization pushes weight norms to infinity $(\|\mathbf{w}\| \to \infty)$ to drive loss to $0$.

* **Proof**:
  Assume a linearly separable dataset where for all $y_i = 1$, $z_i = \mathbf{w}^T \mathbf{x}_i + b > 0$.
  The sample log loss is:
  $$\ell_i(\mathbf{w}) = -\ln(\sigma(z_i)) = \ln(1 + e^{-z_i})$$
  Scale the weight vector by a scalar constant $k > 1$, such that $\mathbf{w}' = k\mathbf{w}$ and $b' = kb$:
  $$\ell_i(k\mathbf{w}) = \ln(1 + e^{-k z_i})$$
  Taking the limit as $k \to \infty$:
  $$\lim_{k \to \infty} \ell_i(k\mathbf{w}) = \lim_{k \to \infty} \ln(1 + e^{-k z_i}) = \ln(1 + 0) = 0$$
  Because the empirical loss is strictly convex with no finite minimum at $\ell = 0$, gradient descent indefinitely inflates $\|\mathbf{w}\| \to \infty$, driving predicted probabilities asymptotically to binary extremes $\{0, 1\}$. $\blacksquare$

## 4. Loss Landscapes: Log Loss vs. Squared Error & Precision Issues

### 4.1 Non-Convexity of Squared Error with Sigmoid Activations
Pairing mean squared error (MSE) $L = \frac{1}{2}(\hat{y} - y)^2$ with a sigmoid activation $\hat{y} = \sigma(z)$ produces a non-convex objective with respect to $\mathbf{w}$:
$$\frac{\partial L}{\partial w_j} = (\hat{y} - y) \cdot \sigma'(z) \cdot x_j = (\hat{y} - y) \hat{y}(1 - \hat{y}) x_j$$
When a prediction is severely incorrect (e.g., $y=1$ but $z \to -\infty \implies \hat{y} \to 0$), $\sigma'(z) = \hat{y}(1 - \hat{y}) \to 0$. This induces vanishing gradients and plateaus.

In contrast, Log Loss cancels the sigmoid derivative:
$$\frac{\partial L_{LogLoss}}{\partial w_j} = (\hat{y} - y) x_j$$
This guarantees convexity and provides error-proportional gradient steps.

### 4.2 Floating-Point Precision and Numerical Stability
As predictions approach boundary conditions ($\hat{y} \to 0$ or $\hat{y} \to 1$), squared loss computes differences $(1 - \hat{y})^2$. For $\hat{y} = 1 - 10^{-15}$, this yields $10^{-30}$. In standard single-precision IEEE 754 float32 (24-bit mantissa $\approx 7$ significant decimal figures), precision degrades due to catastrophic cancellation ($1.0 - 10^{-15} = 1.0$), demanding higher precision representation (float64, float128).

Log Loss implementations bypass this through the numerically stable Log-Sum-Exp formulation directly on logits:
$$\ln(1 + e^{-z}) = \max(0, -z) + \ln\left(1 + e^{-|z|}\right)$$

## 5. Regularization & Complexity Control

### 5.1 Regularization
Structural constraints or penalties added to the optimization objective to restrict parameter capacity and prevent overfitting.

### 5.2 Key Strategies
1. **$L_2$ Regularization (Ridge / Weight Decay)**:
   $$\mathcal{L}_{reg}(\mathbf{w}) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \right] + \frac{\lambda}{2} \sum_{j=1}^d w_j^2$$
   Imposes a quadratic penalty on weight magnitudes, counteracting the asymptotic divergence demonstrated in Proof 3.
2. **Early Stopping**:
   Tracks validation loss throughout iterations, terminating optimization when validation loss diverges, regardless of continuing decreases in training loss.

## 6. Log Loss Formula, Variable Denotations, and Flowchart

### 6.1 Formula
$$\mathcal{L}(\mathbf{w}) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \right]$$

### 6.2 Variable Denotations
* $N$: Total number of samples in the batch/dataset.
* $y_i \in \{0, 1\}$: Ground truth binary label for sample $i$.
* $\hat{y}_i \in (0, 1)$: Predicted probability that $y_i = 1$, where $\hat{y}_i = \sigma(z_i)$.
* $\ln(\cdot)$: Natural logarithm.

### 6.3 Flowchart
```text
   [ Feature Inputs: x1, x2, ... xn ]
                   │
                   ▼
     Linear Combination (z)
    z = b + w1*x1 + ... + wn*xn  <── (Log Odds)
                   │
                   ▼
        Sigmoid Function σ(z)
         y_hat = 1 / (1 + e^-z)  <── (Predicted Probability ∈ (0,1))
                   │
                   ▼
         Branch by True Label y
          /                 \
     If y = 1             If y = 0
       /                     \
  Loss = -ln(y_hat)      Loss = -ln(1 - y_hat)
       \                     /
        \                   /
         ▼                 ▼
          Combined Log Loss:
   - [ y*ln(y_hat) + (1-y)*ln(1-y_hat) ]
                   │
                   ▼
          Average Over Batch
      Mean Log Loss = (1/N) * ∑ Loss
```

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y_true, y_pred, eps=1e-15):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

z = np.array([-2.0, 0.0, 2.5])
y_hat = sigmoid(z)
y_true = np.array([0, 0, 1])

print(f"Logits (z): {z}")
print(f"Probabilities (y_hat): {y_hat}")
print(f"Mean Log Loss: {log_loss(y_true, y_hat):.4f}")

## 7. Classification Thresholds & Decision Dynamics

Logistic regression predicts continuous conditional probabilities $\hat{y} = P(y=1|\mathbf{x}) \in (0, 1)$. Converting probabilities into binary decisions requires a decision threshold $\tau$:
$$\hat{C} = \begin{cases} 1, & \text{if } \hat{y} \ge \tau \\ 0, & \text{if } \hat{y} < \tau \end{cases}$$

### Why the Standard $\tau = 0.5$ Often Fails:
1. **Asymmetric Misclassification Costs**:
   * *False Positive (FP)*: Misclassifying a critical business transaction or legitimate email as spam.
   * *False Negative (FN)*: Allowing an unwanted spam email into the inbox.
   If misclassifying a legitimate email is substantially worse than letting spam slip through, threshold $\tau$ must be shifted higher (e.g., $0.90$ or $0.95$).
2. **Severe Class Imbalance**:
   If the base rate of an event is $0.01\%$, setting $\tau = 0.5$ on a noisy probability distribution yields overwhelming numbers of false alarms relative to true detections. Adjusting $\tau$ optimizes the operational trade-off.

## 8. The Confusion Matrix & Class Designations

### 8.1 Confusion Matrix Layout ($2 \times 2$)
A confusion matrix cross-tabulates predicted decisions against ground truth states:

| | **Actual Positive ($y=1$)** | **Actual Negative ($y=0$)** |
|---|---|---|
| **Predicted Positive ($\hat{C}=1$)** | **True Positive (TP)**<br>Correct hit | **False Positive (FP)**<br>Type I Error (False Alarm) |
| **Predicted Negative ($\hat{C}=0$)** | **False Negative (FN)**<br>Type II Error (Missed Detection) | **True Negative (TN)**<br>Correct rejection |

### 8.2 Row Totals vs. Column Totals
* **Row Totals (Predicted Outlines)**:
  * Predicted Positives = $\text{TP} + \text{FP}$
  * Predicted Negatives = $\text{FN} + \text{TN}$
* **Column Totals (Ground Truth Base Rates)**:
  * Actual Positives = $\text{TP} + \text{FN}$
  * Actual Negatives = $\text{FP} + \text{TN}$

### 8.3 Principles for Assigning Positive ($1$) vs. Negative ($0$)
The mathematical assignment is arbitrary, but conventionally assigned using three system criteria:
1. **The Target Detection**: The condition, state, or event under investigation (e.g., fraud, defect, disease, spam).
2. **The Minority / Anomaly Class**: In imbalanced data, the rare class is designated positive so that metrics like Precision ($\frac{\text{TP}}{\text{TP} + \text{FP}}$) and Recall ($\frac{\text{TP}}{\text{TP} + \text{FN}}$) assess detection efficiency on rare cases.
3. **The Action-Triggering State**: The state that initiates an intervention, alert, or downstream processing pipeline.

In [ ]:
import numpy as np

def sigmoid(z):
    """Numerically stable sigmoid function."""
    return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))

def binary_log_loss(y_true, y_pred, eps=1e-15):
    """Compute mean binary cross-entropy loss."""
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred))

def compute_confusion_matrix(y_true, y_pred_prob, threshold=0.5):
    """Compute 2x2 confusion matrix from probabilities and decision threshold."""
    y_pred = (y_pred_prob >= threshold).astype(int)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    return np.array([[tp, fp], [fn, tn]])

# Demonstration
np.random.seed(42)
logits = np.array([-3.2, -1.1, 0.2, 0.8, 1.5, 2.7])
probabilities = sigmoid(logits)
ground_truth = np.array([0, 0, 0, 1, 1, 1])

print(f"Logits: {logits}")
print(f"Predicted Probabilities: {np.round(probabilities, 4)}")
print(f"Mean Log Loss: {binary_log_loss(ground_truth, probabilities):.4f}")
print("\nConfusion Matrix (Threshold = 0.5):")
print(compute_confusion_matrix(ground_truth, probabilities, threshold=0.5))
print("\nConfusion Matrix (Threshold = 0.8):")
print(compute_confusion_matrix(ground_truth, probabilities, threshold=0.8))